# BlackMirror — TRIBE v2 GPU reference run

Runs Meta's TRIBE v2 on a Colab GPU and dumps a **ground-truth capture** of the
model's real output shape, TR, dtype and segment structure.

Two uses:
1. **Verification** — confirm BlackMirror's schemas match a stock CUDA run, since the
   local macOS path applies documented compatibility patches.
2. **Benchmarking** — a GPU timing reference for `docs/phase1_benchmark.md`.

**Runtime → Change runtime type → T4 GPU** before running.

> TRIBE v2 is **CC-BY-NC-4.0 — non-commercial use only**.
> Output is a *predicted* cortical response for an *average* subject, not a measurement.


## 1. Install


In [ ]:
!pip -q install "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"
print('installed — restart the runtime if Colab asks, then continue from section 2')


## 2. HuggingFace access

TRIBE's text encoder is the **gated** `meta-llama/Llama-3.2-3B`. Accept the license at
<https://huggingface.co/meta-llama/Llama-3.2-3B>, then log in.

Content with no speech skips the text encoder entirely and runs without this.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


## 3. Environment


In [ ]:
import torch, platform, sys
print('python      ', sys.version.split()[0])
print('torch       ', torch.__version__)
print('cuda        ', torch.cuda.is_available(), torch.version.cuda)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('gpu         ', props.name, f'{props.total_memory/1e9:.1f} GB')
print('platform    ', platform.platform())


## 4. Load the model


In [ ]:
from pathlib import Path
import time
from tribev2.demo_utils import TribeModel, download_file

CACHE = Path('./cache'); CACHE.mkdir(exist_ok=True)

t0 = time.perf_counter()
model = TribeModel.from_pretrained('facebook/tribev2', cache_folder=CACHE)
load_seconds = time.perf_counter() - t0
print(f'loaded in {load_seconds:.1f}s')

TR = float(model.data.TR)
MESH = model.data.neuro.projection.mesh
print('TR (s)      ', TR)
print('mesh        ', MESH)
print('avg subjects', model.average_subjects)
print('drop empty  ', model.remove_empty_segments)
print('head params ', sum(p.numel() for p in model._model.parameters()))


## 5. Stimulus

The Sintel trailer (Creative Commons) is Meta's own demo clip. Keep it short —
cost scales with duration.


In [ ]:
STIMULUS = CACHE / 'sample.mp4'
download_file('https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4', STIMULUS)

# Trim to 20 s to keep the reference run quick.
TRIMMED = CACHE / 'sample_20s.mp4'
!ffmpeg -y -v error -i "{STIMULUS}" -t 20 -c copy "{TRIMMED}"
print(TRIMMED, TRIMMED.stat().st_size / 1e6, 'MB')


## 6. Preprocess and predict


In [ ]:
t0 = time.perf_counter()
events = model.get_events_dataframe(video_path=str(TRIMMED))
preprocess_seconds = time.perf_counter() - t0

t0 = time.perf_counter()
preds, segments = model.predict(events=events)
inference_seconds = time.perf_counter() - t0

print('preprocess  ', f'{preprocess_seconds:.1f}s')
print('inference   ', f'{inference_seconds:.1f}s')
print('preds       ', preds.shape, preds.dtype)
print('segments    ', len(segments))


## 7. Ground-truth capture

Everything BlackMirror's schemas assume, read from the real run. Download
`tribe_ground_truth.json` and compare it with `docs/tribe_integration.md`.


In [ ]:
import json, numpy as np, hashlib

starts = np.array([s.start for s in segments], dtype=float)
durations = np.array([s.duration for s in segments], dtype=float)
n_events = np.array([len(s.ns_events) for s in segments], dtype=int)
gaps = np.diff(starts) if starts.size > 1 else np.array([])

capture = {
    'model_id': 'facebook/tribev2',
    'tr_seconds': TR,
    'surface_mesh': MESH,
    'average_subjects': bool(model.average_subjects),
    'remove_empty_segments': bool(model.remove_empty_segments),
    'prediction': {
        'shape': list(preds.shape),
        'dtype': str(preds.dtype),
        'nan_count': int(np.isnan(preds).sum()),
        'inf_count': int(np.isinf(preds).sum()),
        'min': float(np.nanmin(preds)), 'max': float(np.nanmax(preds)),
        'mean': float(np.nanmean(preds)), 'std': float(np.nanstd(preds)),
        'p01': float(np.nanpercentile(preds, 1)),
        'p50': float(np.nanpercentile(preds, 50)),
        'p99': float(np.nanpercentile(preds, 99)),
        'temporal_variance_mean': float(np.nanvar(preds, axis=0).mean()),
    },
    'segments': {
        'count': len(segments),
        'attributes': sorted(a for a in dir(segments[0]) if not a.startswith('_')),
        'first_start': float(starts.min()), 'last_stop': float((starts + durations).max()),
        'unique_gaps': sorted({round(float(g), 6) for g in gaps})[:20],
        'timeline_is_contiguous': bool(np.all(np.abs(gaps - TR) <= 1e-3)) if gaps.size else True,
        'n_events_min': int(n_events.min()), 'n_events_max': int(n_events.max()),
        'event_types': sorted({str(getattr(e, 'type', type(e).__name__))
                               for s in segments for e in s.ns_events}),
    },
    'events_table': {
        'n_rows': int(len(events)),
        'columns': [str(c) for c in events.columns],
        'type_counts': {str(k): int(v) for k, v in events.type.value_counts().items()},
    },
    'performance': {
        'model_load_seconds': load_seconds,
        'preprocess_seconds': preprocess_seconds,
        'inference_seconds': inference_seconds,
        'peak_gpu_allocated_bytes': int(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else None,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
    'stimulus_sha256': hashlib.sha256(TRIMMED.read_bytes()).hexdigest(),
    'torch_version': torch.__version__,
}

Path('tribe_ground_truth.json').write_text(json.dumps(capture, indent=2))
print(json.dumps(capture, indent=2))


## 8. Save a small fixture

The full matrix is large. Save a downsampled slice for offline inspection.


In [ ]:
np.savez_compressed(
    'tribe_reference_sample.npz',
    predictions_head=preds[:8].astype('float32'),   # first 8 time points, all vertices
    segment_start_seconds=starts,
    segment_duration_seconds=durations,
    segment_n_events=n_events,
)
print('wrote tribe_reference_sample.npz')

from google.colab import files
files.download('tribe_ground_truth.json')
files.download('tribe_reference_sample.npz')


## 9. Optional — visual sanity check

Not the Phase 2 visualization; just confirmation that the prediction has spatial
structure on the cortex.


In [ ]:
from tribev2.plotting import PlotBrain
plotter = PlotBrain(mesh=MESH)
plotter.plot(preds[0])


---
**Reminder.** These are predicted cortical responses for an average subject.
They do not establish emotion, memory, attention, preference or purchase intent,
and are not for medical use. See `docs/scientific_limitations.md`.
